#### EDA 03: Order items table
**File:** `olist_order_items_dataset.csv`  |  **Goal:** understand the item lines of each order, using this table only. No joins happen here. Cross-table checks are listed at the end and done after every table is explored.

#### Load and first look
**Question:** how big is the table and what does one row stand for?

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

RAW = Path("../../data/raw")                      # path from the notebook folder to the raw data
items_raw = pd.read_csv(RAW / "olist_order_items_dataset.csv")
items = items_raw.copy()                          # all changes happen on the copy, raw stays untouched

print("shape:", items.shape)
items.head(3)

shape: (112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


- 112,650 rows and 7 columns: order, line number, product, seller, shipping deadline, price and freight.
- `order_item_id` is the line number inside an order (1, 2, 3...). It is not an item identifier.
- One row is one item line, not one order. The orders table has 99,441 rows (raw data summary), so this table has more rows than orders.

#### Data types
**Question:** are the columns in a usable type?

In [2]:
print(items.dtypes)

# The shipping deadline is a date stored as text
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])
items["shipping_limit_date"].agg(["min", "max"])

order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object


min   2016-09-19 00:15:34
max   2020-04-09 22:35:08
Name: shipping_limit_date, dtype: datetime64[us]

- All columns load with a sensible type. Only `shipping_limit_date` is text and is converted to a datetime.
- The deadline runs from 2016-09-19 to 2020-04-09. The orders data ends in 2018, so a date in 2020 is suspicious (see the shipping deadline section).

#### Keys and duplicates
**Question:** what identifies a row, and is any row repeated?

In [3]:
print("fully duplicated rows             :", items.duplicated().sum())
print("repeated (order_id, order_item_id):", items.duplicated(["order_id", "order_item_id"]).sum())

# Line numbers inside each order should be 1, 2, ..., n with no gaps
g = items.groupby("order_id")["order_item_id"].agg(["min", "max", "nunique", "size"])
sequence_ok = (g["min"] == 1) & (g["max"] == g["size"]) & (g["nunique"] == g["size"])
print("orders with line numbers 1..n     :", sequence_ok.sum(), "of", len(g))

fully duplicated rows             : 0
repeated (order_id, order_item_id): 0
orders with line numbers 1..n     : 98666 of 98666


- No duplicated rows. The pair (`order_id`, `order_item_id`) is unique, so it is the key of this table.
- Line numbers run 1 to n without gaps in all 98,666 orders found in this table.
- `order_id` alone is not a key. This is the join trap: merging this table onto orders directly would repeat order rows (tested after the joins).

#### Missing values
**Question:** which columns have gaps?

In [4]:
# Column-wise missing count and percentage, for every column (zeros included)
missing = pd.DataFrame({"n_rows": len(items),
                        "n_missing": items.isna().sum(),
                        "pct_missing": (items.isna().mean() * 100).round(2)})
display(missing.sort_values("n_missing", ascending=False))

,n_rows,n_missing,pct_missing
order_id,112650,0,0.0
order_item_id,112650,0,0.0
product_id,112650,0,0.0
seller_id,112650,0,0.0
shipping_limit_date,112650,0,0.0
price,112650,0,0.0
freight_value,112650,0,0.0


- No column has a missing value. Nothing to impute or flag in this table.

#### Counts inside this table
**Question:** how many distinct orders, products and sellers does the table hold?

In [5]:
print("item lines        :", len(items))
print("distinct orders   :", items["order_id"].nunique())
print("distinct products :", items["product_id"].nunique())
print("distinct sellers  :", items["seller_id"].nunique())

item lines        : 112650
distinct orders   : 98666
distinct products : 32951
distinct sellers  : 3095


- 112,650 lines belong to 98,666 distinct orders, 32,951 products and 3,095 sellers.
- The orders table has 99,441 rows (raw data summary), so up to 775 orders may have no item lines at all.
- Whether those orders exist, and whether every product and seller id exists in its own table, is checked after the joins.

#### Lines per order
**Question:** how many lines does an order have?

In [6]:
lines_per_order = items.groupby("order_id").size()

# Group 6 and more together for display
bucket = lines_per_order.clip(upper=6)
dist = bucket.value_counts().sort_index().rename("n_orders").to_frame()
dist.index = [str(i) if i < 6 else "6+" for i in dist.index]
dist.index.name = "lines_per_order"
dist["pct_orders"] = (dist["n_orders"] / len(lines_per_order) * 100).round(2)
display(dist)

multi = lines_per_order[lines_per_order > 1]
print("orders with more than 1 line :", len(multi), f"({len(multi) / len(lines_per_order) * 100:.1f}% of orders in this table)")
print("lines inside those orders    :", multi.sum(), f"({multi.sum() / len(items) * 100:.1f}% of all lines)")
print("maximum lines in one order   :", lines_per_order.max())

,n_orders,pct_orders
lines_per_order,,
1,88863,90.06
2,7516,7.62
3,1322,1.34
4,505,0.51
5,204,0.21
6+,256,0.26


orders with more than 1 line : 9803 (9.9% of orders in this table)
lines inside those orders    : 23787 (21.1% of all lines)
maximum lines in one order   : 21


- 88,863 orders (90.1%) have a single line.
- 9,803 orders (9.9%) have more than one line, but they hold 23,787 lines, which is 21.1% of all lines.
- The largest order has 21 lines.
- Any statistic computed on lines (average price, share of freight) gives bigger orders more weight than an order-level statistic does.

#### Quantity: lines vs distinct products
**Question:** the table has no quantity column. Do repeated lines mean the same product bought several times?

In [7]:
same_line = items.groupby(["order_id", "product_id", "seller_id"]).size()
print("distinct (order, product, seller) combinations:", len(same_line), "| total lines:", len(items))
print("combinations that repeat (quantity > 1)        :", (same_line > 1).sum())
print("lines inside those repeated combinations       :", same_line[same_line > 1].sum())

products_per_order = items.groupby("order_id")["product_id"].nunique()
print("\norders with more than 1 line            :", (lines_per_order > 1).sum())
print("orders with more than 1 distinct product:", (products_per_order > 1).sum())

distinct (order, product, seller) combinations: 102425 | total lines: 112650
combinations that repeat (quantity > 1)        : 7088
lines inside those repeated combinations       : 17313

orders with more than 1 line            : 9803
orders with more than 1 distinct product: 3236


- There are 112,650 lines but only 102,425 distinct (order, product, seller) combinations.
- In 7,088 combinations the same product from the same seller repeats, covering 17,313 lines. A repeated line works as a quantity.
- 9,803 orders have more than one line, but only 3,236 have more than one distinct product. Most multi-line orders are the same product repeated.
- **Decision:** keep both numbers at order level: `n_lines` (units) and `n_products` (distinct products). They answer different questions.

#### Sellers per order
**Question:** can an order have more than one seller? This decides how to answer "which seller is behind this order".

In [8]:
sellers_per_order = items.groupby("order_id")["seller_id"].nunique()
seller_dist = sellers_per_order.value_counts().sort_index().rename("n_orders").to_frame()
seller_dist.index.name = "sellers_per_order"
seller_dist["pct_orders"] = (seller_dist["n_orders"] / len(sellers_per_order) * 100).round(2)
display(seller_dist)

# Seller value share inside each order (price only)
seller_value = items.groupby(["order_id", "seller_id"], as_index=False)["price"].sum()
seller_value["share"] = seller_value["price"] / seller_value.groupby("order_id")["price"].transform("sum")

multi_ids = sellers_per_order[sellers_per_order > 1].index
top_share = seller_value[seller_value["order_id"].isin(multi_ids)].groupby("order_id")["share"].max()
print("multi-seller orders                                          :", len(multi_ids))
print("... where the biggest seller holds 50% or more of the value :", f"{(top_share >= 0.5).mean() * 100:.1f}%")

,n_orders,pct_orders
sellers_per_order,,
1,97388,98.70
2,1219,1.24
3,54,0.05
4,3,0.00
5,2,0.00


multi-seller orders                                          : 1278
... where the biggest seller holds 50% or more of the value : 97.5%


- 97,388 orders (98.7%) have a single seller, so the seller is clear for almost all orders.
- 1,278 orders (1.3%) have 2 to 5 sellers.
- In 97.5% of those, one seller holds at least half of the order value.
- **Decision:** define `primary_seller_id` as the seller with the largest share of the order value (ties broken by `seller_id`), and keep `n_sellers` as a separate feature. Seller history in Part 2 is attached to the primary seller.

#### Price and freight
**Question:** what do price and freight look like, and are there impossible values?

In [9]:
items["freight_ratio"] = items["freight_value"] / items["price"]       # freight as a share of the item price

items[["price", "freight_value", "freight_ratio"]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.round(2)

,count,mean,std,min,1%,5%,50%,95%,99%,max
price,112650.0,120.65,183.63,0.85,9.99,17.00,74.99,349.90,890.00,6735.00
freight_value,112650.0,19.99,15.81,0.00,4.42,7.78,16.26,45.12,84.52,409.68
freight_ratio,112650.0,0.32,0.35,0.00,0.02,0.05,0.23,0.87,1.55,26.24


In [10]:
print("lines with price <= 0       :", (items["price"] <= 0).sum())
print("lines with freight == 0     :", (items["freight_value"] == 0).sum(), f"({(items['freight_value'] == 0).mean() * 100:.2f}%)")
print("lines with freight > price  :", (items["freight_value"] > items["price"]).sum(), f"({(items['freight_value'] > items['price']).mean() * 100:.1f}%)")

lines with price <= 0       : 0
lines with freight == 0     : 383 (0.34%)
lines with freight > price  : 4124 (3.7%)


- Price: median 74.99, mean 120.65, 99th percentile 890, maximum 6,735. The mean is far above the median, so the distribution is strongly right-skewed.
- Freight: median 16.26, mean 19.99, maximum 409.68.
- Freight ratio: median 0.23 and the 95th percentile 0.87, so freight is usually about a quarter of the price but can approach it. The maximum is 26.24.
- No price is zero or negative. 383 lines (0.34%) have free freight. 4,124 lines (3.7%) have freight higher than the price.
- Free freight is a real case (a promotion or free shipping), not a missing value.

#### Outliers in price and freight
**Question:** are the extreme values real?

In [11]:
print("lines with price > 1000     :", (items["price"] > 1000).sum())
print("lines with price > 2000     :", (items["price"] > 2000).sum())
print("lines with freight > 100    :", (items["freight_value"] > 100).sum())
print("lines with freight ratio > 5:", (items["freight_ratio"] > 5).sum())

print("\nlines with the highest freight ratio:")
items.nlargest(5, "freight_ratio")[["order_id", "price", "freight_value", "freight_ratio"]].round(2)

lines with price > 1000     : 844
lines with price > 2000     : 123
lines with freight > 100    : 671
lines with freight ratio > 5: 41

lines with the highest freight ratio:


,order_id,price,freight_value,freight_ratio
87081,c5bdd8ef3c0ec420232e668302179113,0.85,22.30,26.24
27652,3ee6513ae7ea23bdfab5b9ab60bffcb5,0.85,18.23,21.45
48625,6e864b3f0ec71031117ad4cf46b7f2a1,0.85,18.23,21.45
110535,fb265b2dc558a56445dfc48f8224e201,9.90,121.22,12.24
94495,d642656598ae928a250620315d19e87e,4.99,37.04,7.42


- 844 lines cost more than 1,000 and 123 more than 2,000. High prices are real products (the maximum is 6,735), not errors.
- 671 lines have freight above 100.
- The highest freight ratios come from very cheap items: a price of 0.85 with freight of 22.30 gives 26.24. Only 41 lines have a ratio above 5.
- **Decision:** keep all lines. Freight ratio is unstable for tiny prices, so cap or log it only inside the model pipeline, not in the data.

#### Shipping deadline
**Question:** is the seller's shipping deadline in a believable range, and does it differ between lines of one order?

In [12]:
deadline = items["shipping_limit_date"]
display(deadline.dt.year.value_counts().sort_index().rename("n_lines").to_frame())

after_data = items[deadline.dt.year >= 2019]
print("lines with a deadline in 2019 or later:", len(after_data), "in", after_data["order_id"].nunique(), "orders")
display(after_data[["order_id", "shipping_limit_date", "price"]])

# Do the lines of one order have different deadlines?
spread = items.groupby("order_id")["shipping_limit_date"].agg(["min", "max"])
spread_days = (spread["max"] - spread["min"]).dt.total_seconds() / 86400
print("orders with more than one distinct deadline:", (spread_days > 0).sum())
print("... median / maximum gap between deadlines (days):", round(spread_days[spread_days > 0].median(), 1), "/", round(spread_days.max(), 1))

,n_lines
shipping_limit_date,
2016,370
2017,49765
2018,62511
2020,4


lines with a deadline in 2019 or later: 4 in 3 orders


,order_id,shipping_limit_date,price
8643,13bdf405f961a6deec817d817f5c6624,2020-02-05 03:30:51,69.99
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,2020-02-03 20:23:22,75.99
85729,c2bb89b5c1dd978d507284be78a04cb2,2020-04-09 22:35:08,99.99
85730,c2bb89b5c1dd978d507284be78a04cb2,2020-04-09 22:35:08,99.99


orders with more than one distinct deadline: 336
... median / maximum gap between deadlines (days): 3.0 / 25.0


- Deadlines fall in 2016 (370 lines), 2017 (49,765) and 2018 (62,511), plus 4 lines in 2020.
- The 4 lines in 2020 (3 orders) are after the end of the data (2018). These are errors or manual entries.
- 336 orders have different deadlines on different lines. The median gap is 3.0 days and the maximum is 25.0 days, which fits several sellers or several shipments.
- **Decision:** keep all lines. The deadline is compared with the purchase and approval dates after the joins, because that decides whether it can be a feature.

#### Order-level totals (inside this table only)
**Question:** what does one row per order look like when item lines are combined? This step uses only the items table. No other table is joined.

In [13]:
primary = (seller_value.sort_values(["order_id", "share", "seller_id"], ascending=[True, False, True])
           .groupby("order_id").head(1).set_index("order_id")["seller_id"])

order_items = items.groupby("order_id").agg(
    n_lines=("order_item_id", "size"),
    n_products=("product_id", "nunique"),
    n_sellers=("seller_id", "nunique"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
)
order_items["freight_ratio"] = order_items["total_freight"] / order_items["total_price"]
order_items["primary_seller_id"] = primary

print("order-level table:", order_items.shape, "| order_id unique:", order_items.index.is_unique)
display(order_items[["total_price", "total_freight", "freight_ratio"]].describe(percentiles=[0.5, 0.95, 0.99]).round(2))

print("orders with total_price > 1000 :", (order_items["total_price"] > 1000).sum())
print("orders with total_price > 5000 :", (order_items["total_price"] > 5000).sum())
print("orders with free freight       :", (order_items["total_freight"] == 0).sum())
print("orders with freight > item value:", (order_items["freight_ratio"] > 1).sum(), f"({(order_items['freight_ratio'] > 1).mean() * 100:.1f}%)")
order_items.head(3).round(2)

order-level table: (98666, 7) | order_id unique: True


,total_price,total_freight,freight_ratio
count,98666.00,98666.00,98666.00
mean,137.75,22.82,0.31
std,210.65,21.65,0.31
min,0.85,0.00,0.00
50%,86.90,17.17,0.22
95%,399.90,54.96,0.84
99%,998.90,104.98,1.46
max,13440.00,1794.96,21.45


orders with total_price > 1000 : 943
orders with total_price > 5000 : 6
orders with free freight       : 338
orders with freight > item value: 3159 (3.2%)


,n_lines,n_products,n_sellers,total_price,total_freight,freight_ratio,primary_seller_id
order_id,,,,,,,
00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.9,13.29,0.23,48436dade18ac8b2bce089ec2a041202
00018f77f2f0320c557190d7a144bdd3,1,1,1,239.9,19.93,0.08,dd7ddc04e1b6c2c614352b383efe2d36
000229ec398224ef6ca0657da4fc703e,1,1,1,199.0,17.87,0.09,5b51032eddd242adc84c38acab88f23d


- The 112,650 lines reduce to 98,666 order rows, with a unique `order_id`. One row per order is ready to be joined later.
- Order value: median 86.90, mean 137.75, 99th percentile 998.90, maximum 13,440. 943 orders are above 1,000 and 6 above 5,000.
- Order-level freight ratio: median 0.22, 95th percentile 0.84, maximum 21.45. 3,159 orders (3.2%) pay more freight than the value of the items, and 338 have free freight.
- The row count after merging this table onto orders (expected 99,441) is tested after the joins.

#### Seller and product concentration
**Question:** how concentrated are sales across sellers and products? This tells how much history a seller or product has for Part 2.

In [14]:
seller_lines = items.groupby("seller_id").size().sort_values(ascending=False)
print("sellers in the items table        :", len(seller_lines))
print("top 10 sellers, share of lines    :", f"{seller_lines.head(10).sum() / len(items) * 100:.1f}%")
print("top 20% of sellers, share of lines:", f"{seller_lines.head(int(len(seller_lines) * 0.2)).sum() / len(items) * 100:.1f}%")
print("sellers with 5 lines or fewer     :", (seller_lines <= 5).sum(), f"({(seller_lines <= 5).mean() * 100:.1f}% of sellers)")

product_lines = items.groupby("product_id").size()
print("\nproducts in the items table       :", len(product_lines))
print("products sold in only 1 line      :", (product_lines == 1).sum(), f"({(product_lines == 1).mean() * 100:.1f}% of products)")
print("most lines for one product        :", product_lines.max())

sellers in the items table        : 3095
top 10 sellers, share of lines    : 14.2%
top 20% of sellers, share of lines: 82.6%
sellers with 5 lines or fewer     : 1343 (43.4% of sellers)

products in the items table       : 32951
products sold in only 1 line      : 18117 (55.0% of products)
most lines for one product        : 527


- All 3,095 sellers appear. The top 10 sellers hold 14.2% of lines, and the top 20% of sellers hold 82.6%. Sales are concentrated in a minority of sellers.
- 1,343 sellers (43.4%) have 5 lines or fewer. For these, a seller history built from past orders will be thin or empty at the start.
- 32,951 products, and 18,117 (55.0%) sell in only one line. A product id carries almost no history, so product category is the better feature than product id.

#### Summary
**The story in short.** The items table is clean (no missing values, no duplicates), but it is not one row per order: 112,650 lines belong to 98,666 orders, and 21.1% of lines sit in the 9.9% of orders with more than one line. There is no quantity column, so repeated lines act as quantity. Almost all orders have one seller (98.7%), and a few have several. Price and freight are right-skewed with real extreme values, and the shipping deadline has a few clear errors. Seller history will be thin for many small sellers.

**Data quality issues**

| # | Issue | Records | What it is | Action | Cost of the alternative |
|---|---|---|---|---|---|
| 1 | Several lines per order | 9,803 orders, 23,787 lines | By design | Reduce to one row per order before any join | A direct merge repeats order rows |
| 2 | No quantity column | 7,088 repeating combinations (17,313 lines) | Repeated lines act as quantity | Keep `n_lines` and `n_products` | Using only one of them hides either units or variety |
| 3 | Orders with several sellers | 1,278 | Seller is unclear | `primary_seller_id` = largest value share, plus `n_sellers` | Picking the first line is arbitrary |
| 4 | Free freight and freight above price | 383 and 4,124 lines | Real cases (promotions, cheap items) | Keep, cap or log the ratio in the model only | Dropping removes real behaviour |
| 5 | Extreme prices and freight ratios | 844 lines over 1,000, 41 lines with ratio over 5 | Real products and very cheap items | Keep, cap in the pipeline | Removing loses the high-value orders |
| 6 | Deadlines after the end of the data | 4 lines (3 orders) in 2020 | Errors or manual entries | Keep, cap any deadline feature in the pipeline | Dropping loses lines of valid orders |
| 7 | Thin seller history | 1,343 sellers with 5 lines or fewer | Many small sellers | Plan for empty history in Part 2 | Treating thin history as reliable gives noisy rates |

**Checks planned after the joins** (not done here on purpose)
- Orders without item lines (up to 775): who they are, and whether "has items" is known at approval or only after the order failed.
- Row count when items are merged onto orders directly, against the aggregated order table.
- Every item `order_id`, `product_id` and `seller_id` against its own table.
- Shipping deadline against purchase and approval dates.
- Basket features (lines, sellers, value, freight ratio) against the bad-review rate.

**What can be used at order approval (for Part 2)**

| Field | Known at approval? | Note |
|---|---|---|
| `n_lines`, `n_products`, `n_sellers`, `total_price`, `total_freight`, `freight_ratio` | Yes | Built from the basket at purchase |
| `primary_seller_id` | Yes | Key for seller history. History must use only orders closed before this one. |
| `price` of a line | Yes | Paid at checkout |
| `shipping_limit_date` | To be checked | Looks like a rule set from the purchase, not provable inside this table |
| Existence of item lines | To be checked | May only be known after an order failed, so it may leak the outcome |